# Model 2 — DistilBERT Sequence Ranker

**Project:** Smart MCQ Solver · DL & GenAI · Milestone 3  
**Kernel path:** `nb/pretrained/pre_trained.ipynb`  

---

## Architecture

Each MCQ question carries five candidate options (A–E). This notebook frames the task as **pairwise binary scoring**:

| Step | What happens |
|------|--------------|
| **Retrieval** | FAISS (cosine via inner-product) over MiniLM-L6-v2 embeddings returns the Top-3 relevant passages for a prompt |
| **Input format** | `[CLS] <context> [SEP] <question> + <option text> [SEP]` — one sequence per option |
| **Scoring** | `distilbert-base-uncased` + single linear head outputs one logit per (context, question+option) pair |
| **Loss** | `BCEWithLogitsLoss` — correct option labelled 1, four distractors labelled 0 |
| **Prediction** | Five raw logits sorted descending; top-3 option letters → `submission.csv` |

> **Local safety rule:** `DEVICE = cpu` and `SMOKE_TEST = True` are active by default.  
> Switch both on Kaggle before `kaggle kernels push`.

## 1. Library Imports

In [ ]:
# Install faiss if missing (Kaggle runtime check)
import subprocess
import sys
try:
    import faiss
except ImportError:
    print('Installing faiss-cpu...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faiss-cpu'])
    import faiss
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Hugging Face — all weights loaded locally, zero external API calls
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

import wandb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print("Imports ready.")

## 2. Dynamic Path Resolver

Notebook lives two levels deep (`nb/pretrained/`), so the local data path is `../../data/`.  
On Kaggle the competition data mounts at `/kaggle/input/smart-mcq-solver-challenge/`.

> **Viva Defence:** `os.path.exists` is evaluated at runtime, so no environment-specific edits are ever needed.

In [ ]:
KAGGLE_INPUT = "/kaggle/input/competitions/smart-mcq-solver-challenge"

def get_path(filename: str) -> str:
    """Resolve a data filename across local and Kaggle environments.

    Resolution order:
      1. ../../data/   — local dev when CWD is nb/pretrained/
      2. data/         — local dev when CWD is the project root
      3. Kaggle mount  — production GPU kernel
    """
    candidates = [
        os.path.join("..", "..", "data", filename),
        os.path.join("data", filename),
        os.path.join(KAGGLE_INPUT, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"'{filename}' not found in any of:\n" + "\n".join(f"  {c}" for c in candidates)
    )


trn_df = pd.read_csv(get_path("train.csv"))
tst_df = pd.read_csv(get_path("test.csv"))

print(f"train.csv  shape : {trn_df.shape}")
print(f"test.csv   shape : {tst_df.shape}")
print(f"Columns          : {list(trn_df.columns)}")
trn_df.head(2)

## 3. W&B Initialization & Hyperparameter Config

> **Viva Defence:** All hyperparameters live in `wandb.config`. Changing a value and re-running automatically creates a new tracked run — no manual rename needed.

In [ ]:
# Authenticate W&B — runs land in project: 23f2004343-t22026
wandb.login(key="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", relogin=True)

run = wandb.init(
    project="23f2004343-t22026",
    name="Model_2_Pretrained",
    config={
        # model
        "base_model"       : "distilbert-base-uncased",
        "num_labels"       : 1,           # single logit → BCEWithLogitsLoss
        "max_seq_length"   : 512,
        # retrieval
        "embed_model"      : "sentence-transformers/all-MiniLM-L6-v2",
        "retrieval_top_k"  : 3,
        "chunk_size_words" : 80,
        "chunk_stride"     : 40,
        # training
        "learning_rate"    : 2e-5,
        "epochs"           : 3,
        "batch_size"       : 8,
        "warmup_ratio"     : 0.1,
        "optimizer"        : "AdamW",
        "loss_fn"          : "BCEWithLogitsLoss",
    },
)
cfg = wandb.config

# ── Compute device ──────────────────────────────────────────────────────────
# ⚠️  LOCAL SAFETY: CPU forced — swap for auto-detect on Kaggle:
#   DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE = torch.device("cpu")
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"W&B run    : {run.name}  |  project : {run.project}")
print(f"Device     : {DEVICE}")
print(f"Config     : {dict(cfg)}")

## 4. Text Chunking & FAISS Context Index (MiniLM-L6-v2)

Each training row becomes a passage (prompt + all five options). Overlapping word-level chunks are embedded with `all-MiniLM-L6-v2` (384-dim) and stored in a `faiss.IndexFlatIP` for exact cosine search.

> **Viva Defence:** L2-normalised embeddings turn inner-product search into cosine similarity. `IndexFlatIP` is brute-force but exact — fast enough for ~20 K chunks.

In [ ]:
CHUNK_WORDS  = int(cfg.chunk_size_words)
CHUNK_STRIDE = int(cfg.chunk_stride)
TOP_K        = int(cfg.retrieval_top_k)

# ── Text helpers ─────────────────────────────────────────────────────────────
def clean(raw: str) -> str:
    t = re.sub(r'<[^>]+>', ' ', str(raw))
    return re.sub(r'\s+', ' ', t).strip().lower()


def row_to_passage(row: pd.Series) -> str:
    return ' '.join([
        clean(row['prompt']),
        f"option a: {clean(row['A'])}",
        f"option b: {clean(row['B'])}",
        f"option c: {clean(row['C'])}",
        f"option d: {clean(row['D'])}",
        f"option e: {clean(row['E'])}",
    ])


def sliding_chunks(text: str, size: int, stride: int) -> list:
    words, out, s = text.split(), [], 0
    while s < len(words):
        e = min(s + size, len(words))
        out.append(' '.join(words[s:e]))
        if e == len(words): break
        s += stride
    return out


# ── Resolve saved index paths ─────────────────────────────────────────────────
def _first_existing(paths):
    return next((p for p in paths if os.path.exists(p)), None)

index_path = _first_existing([
    os.path.join("..", "..", "data", "mcq_faiss.index"),
    os.path.join("data", "mcq_faiss.index"),
])
meta_path = _first_existing([
    os.path.join("..", "..", "data", "chunk_meta.pkl"),
    os.path.join("data", "chunk_meta.pkl"),
])

if index_path and meta_path:
    print(f"Loading pre-built FAISS index: {index_path}")
    faiss_index = faiss.read_index(index_path)
    with open(meta_path, "rb") as fh:
        _meta       = pickle.load(fh)
    chunk_texts = _meta["texts"]
    chunk_ids   = _meta["row_ids"]
else:
    # ── Inline rebuild when data_pipeline.ipynb has not been run ──────────────
    print("Index not found — building inline. Run data_pipeline.ipynb first on Kaggle.")
    corpus = []
    for _, row in trn_df.iterrows():
        for ch in sliding_chunks(row_to_passage(row), CHUNK_WORDS, CHUNK_STRIDE):
            corpus.append((ch, int(row['id'])))

    chunk_texts = [c[0] for c in corpus]
    chunk_ids   = [c[1] for c in corpus]

    _builder = SentenceTransformer(cfg.embed_model, device="cpu")
    vecs = _builder.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype(np.float32)

    faiss_index = faiss.IndexFlatIP(vecs.shape[1])
    faiss_index.add(vecs)

    save_dir = os.path.join("..", "..", "data") if os.path.isdir(
        os.path.join("..", "..", "data")
    ) else "data"
    os.makedirs(save_dir, exist_ok=True)
    faiss.write_index(faiss_index, os.path.join(save_dir, "mcq_faiss.index"))
    with open(os.path.join(save_dir, "chunk_meta.pkl"), "wb") as fh:
        pickle.dump({"texts": chunk_texts, "row_ids": chunk_ids}, fh)

# ── Query encoder (always CPU for local safety) ───────────────────────────────
query_encoder = SentenceTransformer(cfg.embed_model, device="cpu")
print(f"FAISS index ready — {faiss_index.ntotal:,} vectors  dim={faiss_index.d}")


def retrieve_context(query: str, k: int = TOP_K) -> str:
    """Return top-k relevant chunks joined as a single context string.

    Viva Defence: query is embedded in the same L2-normalised space as the
    index, so inner-product == cosine similarity. FAISS returns hits sorted
    by descending score automatically.
    """
    q_vec = query_encoder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)                          # shape (1, 384)
    _, hit_idx = faiss_index.search(q_vec, k)     # shape (1, k)
    return " ".join(chunk_texts[i] for i in hit_idx[0] if i != -1)


# Sanity probe
print("Sample retrieval:", retrieve_context(tst_df.iloc[0]['prompt'])[:150], "...")

## 5. Dataset & Tokeniser — `[CLS] Context [SEP] Q + Option [SEP]`

Each question expands to **5 records** (one per option). The tokeniser inserts `[CLS]` and `[SEP]` automatically when a text-pair is supplied.

> **Viva Defence:** DistilBERT's `[CLS]` representation is a learned aggregate of the full sequence — ideal for binary classification. `truncation=True` trims from the right so the most informative beginning of the context is preserved.

In [ ]:
OPTIONS  = ['A', 'B', 'C', 'D', 'E']
MAX_LEN  = int(cfg.max_seq_length)

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
print(f"Tokeniser : {cfg.base_model}  |  vocab size : {tokenizer.vocab_size:,}")

# ── SMOKE TEST TOGGLE ─────────────────────────────────────────────────────────
# ⚠️  Set SMOKE_TEST = False on Kaggle for full-dataset training
SMOKE_TEST = False  # Full dataset — Kaggle GPU kernel
N_ROWS     = 5 if SMOKE_TEST else len(trn_df)


def build_records(df: pd.DataFrame, has_labels: bool = True) -> list:
    """Expand each question row into 5 (context, q+option, label) records.

    Label is 1 for the correct option, 0 for every distractor.
    During test inference `has_labels=False` so the label field is filled with -1.
    """
    records = []
    for _, row in df.iterrows():
        ctx     = retrieve_context(str(row['prompt']))
        correct = str(row['answer']).strip().upper() if has_labels else None
        for opt in OPTIONS:
            records.append({
                "row_id" : int(row['id']),
                "option" : opt,
                "seg_a"  : ctx,
                "seg_b"  : f"{row['prompt']} {row[opt]}",
                "label"  : 1 if opt == correct else 0,
            })
    return records


class MCQRankerDataset(Dataset):
    """Wraps (context, question+option) pairs for DistilBERT sequence classification.

    Tokenisation is on-the-fly to avoid holding all MAX_LEN tensors in RAM.
    """

    def __init__(self, records: list, tok, max_len: int):
        self.records = records
        self.tok     = tok
        self.max_len = max_len

    def __len__(self): return len(self.records)

    def __getitem__(self, idx: int) -> dict:
        r   = self.records[idx]
        enc = self.tok(
            r["seg_a"],             # [CLS] context [SEP]
            r["seg_b"],             # question + option [SEP]
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),        # (MAX_LEN,)
            "attention_mask": enc["attention_mask"].squeeze(0),   # (MAX_LEN,)
            "label"         : torch.tensor(r["label"], dtype=torch.float),
        }


print(f"Building records from {N_ROWS} training rows ({'smoke test' if SMOKE_TEST else 'full'})...")
all_records = build_records(trn_df.head(N_ROWS), has_labels=True)
print(f"Pairs  : {len(all_records)}  |  positives : {sum(r['label'] for r in all_records)}")

trn_recs, val_recs = train_test_split(all_records, test_size=0.1, random_state=SEED)

BATCH      = int(cfg.batch_size)
trn_loader = DataLoader(MCQRankerDataset(trn_recs, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(MCQRankerDataset(val_recs, tokenizer, MAX_LEN), batch_size=BATCH, shuffle=False)

probe = next(iter(trn_loader))
print(f"input_ids : {probe['input_ids'].shape}  |  labels : {probe['label'].tolist()}")

## 6. Model, Optimiser & Scheduler

In [ ]:
# DistilBERT + 1-unit classification head
# The [CLS] hidden state (dim 768) is projected to a scalar logit by the head
ranker = AutoModelForSequenceClassification.from_pretrained(
    cfg.base_model, num_labels=1
)
ranker.to(DEVICE)

n_params = sum(p.numel() for p in ranker.parameters() if p.requires_grad)
print(f"Model : {cfg.base_model}  |  trainable params : {n_params:,}")

criterion = nn.BCEWithLogitsLoss()
optimiser = AdamW(ranker.parameters(), lr=float(cfg.learning_rate), eps=1e-8)

total_steps  = len(trn_loader) * int(cfg.epochs)
warmup_steps = int(total_steps * float(cfg.warmup_ratio))
scheduler    = get_linear_schedule_with_warmup(
    optimiser,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"Steps : {total_steps}  |  warmup : {warmup_steps}")

## 7. Training Loop

> **Viva Defence:**
> - `zero_grad()` clears accumulated gradients before each batch.
> - `backward()` computes gradients via auto-differentiation through the full compute graph.
> - `clip_grad_norm_(1.0)` prevents exploding gradients — a known issue in deep transformer fine-tuning.
> - The scheduler applies linear LR decay after the warmup window, which stabilises convergence.

In [ ]:
def run_eval(model, loader) -> dict:
    """One validation pass — returns loss, accuracy, and macro F1."""
    model.eval()
    total_loss, all_preds, all_labs = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(DEVICE)
            masks = batch["attention_mask"].to(DEVICE)
            labs  = batch["label"].to(DEVICE)
            # logits: (batch, 1) → squeeze → (batch,)
            logits      = model(input_ids=ids, attention_mask=masks).logits.squeeze(-1)
            total_loss += criterion(logits, labs).item()
            preds = (torch.sigmoid(logits) > 0.5).long().cpu().tolist()
            all_preds.extend(preds)
            all_labs.extend(labs.long().cpu().tolist())
    n = max(len(loader), 1)
    return {
        "val_loss"     : total_loss / n,
        "val_accuracy" : accuracy_score(all_labs, all_preds),
        "val_f1_macro" : f1_score(all_labs, all_preds, average="macro", zero_division=0),
    }


best_val_loss = float('inf')
EPOCHS        = int(cfg.epochs)

for epoch in range(1, EPOCHS + 1):
    ranker.train()
    running_loss = 0.0

    for batch in trn_loader:
        ids   = batch["input_ids"].to(DEVICE)
        masks = batch["attention_mask"].to(DEVICE)
        labs  = batch["label"].to(DEVICE)

        optimiser.zero_grad()
        logits = ranker(input_ids=ids, attention_mask=masks).logits.squeeze(-1)
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(ranker.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        running_loss += loss.item()

    avg_trn = running_loss / max(len(trn_loader), 1)
    val_m   = run_eval(ranker, val_loader)

    wandb.log({"epoch": epoch, "train_loss": avg_trn, **val_m})
    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"trn_loss={avg_trn:.4f}  "
        f"val_loss={val_m['val_loss']:.4f}  "
        f"val_acc={val_m['val_accuracy']:.4f}  "
        f"val_f1={val_m['val_f1_macro']:.4f}"
    )

    if val_m['val_loss'] < best_val_loss:
        best_val_loss = val_m['val_loss']
        ckpt_dir = (
            os.path.join("..", "..", "data")
            if os.path.isdir(os.path.join("..", "..", "data"))
            else "data"
        )
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.save(ranker.state_dict(), os.path.join(ckpt_dir, "distilbert_ranker.pt"))
        print(f"  ✓ Checkpoint saved (val_loss={best_val_loss:.4f})")

wandb.run.summary["best_val_loss"] = best_val_loss
print("\nTraining complete.")

## 8. MAP@3 Evaluation + Submission Export

MAP@3 rewards placing the correct answer **earlier** in the ranked prediction list.

```
AP@3 = 1 / rank_of_correct_answer   (if in top-3, else 0)
MAP@3 = mean(AP@3) across all questions
```

> **Viva Defence:** Raw logits are rank-monotone — no sigmoid needed at inference. Five options are scored independently and sorted descending; the top-3 letters become the prediction. The output format `id, A B C` matches the Kaggle submission specification exactly.

In [ ]:
# ── MAP@3 utilities ──────────────────────────────────────────────────────────

def ap_at_3(ranked: list, correct: str) -> float:
    """Average Precision at 3 for one question.

    ranked  : up to 3 predicted option letters, best-first.
    correct : single ground-truth letter.
    """
    hits, score = 0, 0.0
    for k, pred in enumerate(ranked[:3], start=1):
        if pred == correct:
            hits  += 1
            score += hits / k
    return score  # denominator = min(1 relevant, 3) = 1


def map_at_3(pred_map: dict, gt_map: dict) -> float:
    """MAP@3 across all question IDs.

    pred_map : {qid: ['A','C','B'], ...}
    gt_map   : {qid: 'B', ...}
    """
    return float(np.mean([ap_at_3(pred_map[qid], gt_map[qid]) for qid in gt_map]))


# ── Load best checkpoint ─────────────────────────────────────────────────────
ckpt_dir  = (
    os.path.join("..", "..", "data")
    if os.path.isdir(os.path.join("..", "..", "data"))
    else "data"
)
ckpt_path = os.path.join(ckpt_dir, "distilbert_ranker.pt")
if os.path.exists(ckpt_path):
    ranker.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f"Best checkpoint loaded: {ckpt_path}")

ranker.eval()


def score_options(row: pd.Series, context: str) -> dict:
    """Return {option_letter: raw_logit} for all five options of one question."""
    scores = {}
    for opt in OPTIONS:
        enc = tokenizer(
            context,
            f"{row['prompt']} {row[opt]}",
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            scores[opt] = ranker(
                input_ids=enc["input_ids"].to(DEVICE),
                attention_mask=enc["attention_mask"].to(DEVICE),
            ).logits.item()
    return scores


# ── Inference ─────────────────────────────────────────────────────────────────
infer_df = tst_df  # Full test set — all 500 rows

print(f"Scoring {len(infer_df)} test questions...")
rows = []
for _, row in infer_df.iterrows():
    ctx    = retrieve_context(str(row['prompt']))
    scores = score_options(row, ctx)
    # Rank all 5 logits descending; top-3 form the prediction string
    top3   = sorted(scores, key=lambda x: scores[x], reverse=True)[:3]
    rows.append({"id": int(row['id']), "prediction": ' '.join(top3)})

# ── Export ────────────────────────────────────────────────────────────────────
df = pd.DataFrame(rows, columns=["id", "prediction"])
df.to_csv('submission.csv', index=False)

print(f"submission.csv written — {len(df)} rows")
print(df.head())

# Log artifact and close W&B run
artifact = wandb.Artifact("submission_pretrained", type="predictions")
artifact.add_file('submission.csv')
wandb.log_artifact(artifact)
wandb.finish()
print("W&B run closed.")